# Experiment 1 — All models (batch)

Runs **Navigation Element Recognition** (Exp1) for **every** model in `models.json` (typically **12** PaliGemma checkpoints — same list as repo `config/models.json`).

---

## Single-model notebook vs this one

| | Single (`experiment1_evaluation.ipynb`) | This notebook |
|---|--------|---------------|
| Models | **One** (`MODEL_ID`) | **All** (or `RUN_ONLY_IDS` subset) |
| Output | Flat: `*_exp1_*.csv` in `results/exp1/` | Subfolder per model + comparison CSVs |
| VRAM | One load | **Sequential** — unload between models |

---

## Notebook structure

| § | Topic |
|---|--------|
| **1** | Install |
| **2** | **Unzip** images + annotations |
| **3** | **Configuration** — `HF_TOKEN`, paths, `RUN_ONLY_IDS`, `models.json` |
| **4** | Reproducibility JSON |
| **5** | Imports + load questions/answers |
| **6–8** | Metric helpers, plot helpers, per-model runner |
| **9** | **Loop** over all models |
| **10–11** | Comparison tables + cross-model plots |

---

## Before you run (Colab)

1. **Runtime → GPU** — T4+ for 3B; **~40 GB VRAM** for **10B** checkpoints.
2. Upload to **`/content/`**: `images.zip`, `annotations.zip`, **`models.json`** (copy from `config/models.json`).
3. Run **§2 Unzip** → **§3** set `HF_TOKEN` (and optional `RUN_ONLY_IDS`) → **Run all**.
4. Download **`/content/results/exp1/`** as ZIP when finished.

**Partial runs:** `RUN_ONLY_IDS = [5, 6, 7, 8]` — ids must match `"id"` in `models.json`.

---

## Output layout

```
/content/results/exp1/
├── exp1_reproducibility.json
├── <model_safe>/
│   ├── results.csv, aggregate.csv, efficiency.json, matrices.json
│   └── plots/   (11 PNGs per model)
├── all_models_summary.csv
├── all_models_expanded.csv
├── comparison_plots/
└── failed_models.json   (only if a run crashed)
```

**Obstacle safety metrics (aligned with code):** `hazard_miss_rate` on `obstacle_detection` = `1 − recall` (missed obstacles). `safety_hazard_miss_rate` on `obstacle_safety` (path clear) = `FP/(FP+TN)` among actually unsafe cases — lower is better for both.



In [ ]:
# Install dependencies
!pip install -q transformers torch Pillow bert-score accelerate

## 2. Unzip dataset

Upload **`images.zip`** and **`annotations.zip`** to **`/content/`** (Colab sidebar).

Expected after extraction:

- `/content/images/{staircase,elevator,door,door_sign,obstacle}/` — `.jpg` files  
- `/content/annotations/questions.json` and `exp1_expected_answers.json`



In [ ]:
import os
import zipfile

CONTENT = '/content'
ARCHIVES = [
    ('images.zip', 'images'),
    ('annotations.zip', 'annotations'),
]

for zip_name, folder_name in ARCHIVES:
    zip_path = os.path.join(CONTENT, zip_name)
    dest = os.path.join(CONTENT, folder_name)
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(
            f"Missing {zip_path!r}. Upload {zip_name} to /content/."
        )
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest)
    print(f'OK: {zip_name} → {dest}/')

for p in (
    os.path.join(CONTENT, 'annotations', 'questions.json'),
    os.path.join(CONTENT, 'annotations', 'exp1_expected_answers.json'),
):
    if not os.path.isfile(p):
        raise FileNotFoundError(f'Expected file missing after unzip: {p}')

for sub in ('staircase', 'elevator', 'door', 'door_sign', 'obstacle'):
    d = os.path.join(CONTENT, 'images', sub)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.lower().endswith('.jpg')])
        print(f'  images/{sub}/: {n} .jpg files')
    else:
        print(f'  WARNING: missing folder {d}')

print('Unzip complete.')


## 3. Configuration

- **`HF_TOKEN`** — HuggingFace token if models are gated.  
- **`MODELS_JSON_PATH`** — default `/content/models.json` (upload from repo `config/models.json`).  
- **`RUN_ONLY_IDS`** — `[]` = all models; else list of numeric `"id"` from JSON.  
- Requires **§2** unzip complete (`/content/images`, `/content/annotations`).



In [ ]:
# -- Configuration ----------------------------------------------
HF_TOKEN = ''  # Set your HuggingFace token here

IMAGES_PATH      = '/content/images'
ANNOTATIONS_PATH = '/content/annotations'
MODELS_JSON_PATH = '/content/models.json'   # upload config/models.json
RESULTS_PATH     = '/content/results/exp1'

MAX_NEW_TOKENS = 30

# Optional: restrict to a subset of model IDs for partial runs
# Leave empty [] to run all 12 models
RUN_ONLY_IDS = []  # e.g. [5, 6, 7, 8] to run only paligemma2-3b models

# -- Reproducibility seeds --------------------------------------
# With do_sample=False and num_beams=1 pinned in the decoding protocol these
# are no-ops for PaliGemma output, but they are standard hygiene for any
# downstream numpy/torch randomness and let us cite a single seed in the card.
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# cuDNN determinism pinning. Small throughput cost on conv-heavy models in
# exchange for bit-reproducible kernels across repeated runs on the same GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Pinned decoding protocol. Mirrored inside run_inference_timed; kept here so
# the reproducibility card can dump it verbatim without re-deriving anything.
DECODING_PROTOCOL = {
    'do_sample': False,
    'num_beams': 1,
    'temperature': None,
    'max_new_tokens': MAX_NEW_TOKENS,
    'torch_dtype': 'bfloat16 on CUDA, float32 on CPU',
}

# Latency benchmarking protocol:
#   - N_WARMUP inferences per model are discarded before timing (pays cold-start
#     cost once: CUDA kernel compilation, VRAM weight load, cache stabilization).
#   - Each model is then timed N_LATENCY_RUNS times over the full task list.
#   - Reported avg_inference_time_ms = mean over all N_LATENCY_RUNS per-task values.
#   - Quality (exact_match / keyword_recall / bertscore) is computed once from run 0
#     predictions. With the pinned protocol (do_sample=False, num_beams=1) PaliGemma
#     is deterministic across runs, so runs 1..N add no information beyond latency.
N_WARMUP = 3
N_LATENCY_RUNS = 3

print('Config OK')
print(f'Images:      {IMAGES_PATH}')
print(f'Annotations: {ANNOTATIONS_PATH}')
print(f'Results:     {RESULTS_PATH}')
print(f'MAX_NEW_TOKENS: {MAX_NEW_TOKENS}')
print(f'Seed: {SEED}  |  cudnn deterministic: True')
print(f'Latency protocol: N_WARMUP={N_WARMUP}, N_LATENCY_RUNS={N_LATENCY_RUNS}')

## 4. Reproducibility snapshot

Writes `exp1_reproducibility.json` (GPU, library versions, model list). **Requires** `models.json` at `MODELS_JSON_PATH`.



In [ ]:
# -- Reproducibility card ---------------------------------------
import subprocess, datetime, json, os, hashlib
import torch, transformers
from importlib import metadata as importlib_metadata


def get_gpu_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return 'CPU'


def get_git_commit():
    try:
        return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return 'unavailable'


def _pkg_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return 'unavailable'


def _file_sha256(path, chunk_size=65536):
    """SHA-256 of a file, streamed in chunks so large archives do not spike RAM."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


with open(MODELS_JSON_PATH) as f:
    models_config = json.load(f)

all_model_ids = [m['model_id'] for m in models_config['models']]
run_models = (
    [m for m in models_config['models'] if m['id'] in RUN_ONLY_IDS]
    if RUN_ONLY_IDS else models_config['models']
)

# Per-file SHA-256 fingerprints so anyone re-running the experiment can verify
# they are using identical bytes for the dataset, annotations and model config.
DATASET_ARTIFACT_PATHS = [
    '/content/images.zip',
    '/content/annotations/exp1_expected_answers.json',
    '/content/annotations/questions.json',
    '/content/models.json',
]
dataset_artifact_hashes = {}
for p in DATASET_ARTIFACT_PATHS:
    if os.path.isfile(p):
        dataset_artifact_hashes[p] = {
            'sha256': _file_sha256(p),
            'size_bytes': os.path.getsize(p),
        }

LATENCY_PROTOCOL = {
    'n_warmup': N_WARMUP,
    'n_latency_runs': N_LATENCY_RUNS,
    'cuda_synchronize_around_timer': True,
    'warmup_image_source': 'synthetic 224x224 deterministic RGB noise, seed=SEED',
    'image_order': 'shuffled deterministically per run (seed = SEED*1000 + run_idx)',
}

repro = {
    'experiment': 'exp1',
    'date': datetime.datetime.now().isoformat(timespec='seconds'),
    'gpu': get_gpu_name(),
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'pillow_version': _pkg_version('pillow'),
    'bert_score_version': _pkg_version('bert-score'),
    'seed': SEED,
    'cudnn_deterministic': True,
    'max_new_tokens': MAX_NEW_TOKENS,
    'images_path': IMAGES_PATH,
    'annotations_path': ANNOTATIONS_PATH,
    'git_commit': get_git_commit(),
    'models_evaluated': [m['model_id'] for m in run_models],
    'all_models_in_config': all_model_ids,
    'decoding_protocol': DECODING_PROTOCOL,
    'latency_protocol': LATENCY_PROTOCOL,
    'dataset_artifact_hashes': dataset_artifact_hashes,
}

os.makedirs(RESULTS_PATH, exist_ok=True)
repro_path = f'{RESULTS_PATH}/exp1_reproducibility.json'
with open(repro_path, 'w') as f:
    json.dump(repro, f, indent=2)

print('REPRODUCIBILITY CARD')
print('=' * 40)
# Print a compact preview that omits the full hash map (it is in the JSON).
preview = {k: v for k, v in repro.items() if k != 'dataset_artifact_hashes'}
preview['dataset_artifact_hashes'] = f'<{len(dataset_artifact_hashes)} SHA-256 fingerprints saved to JSON>'
for k, v in preview.items():
    print(f'  {k}: {v}')

## 5. Imports and dataset definitions

Loads `questions.json` and `exp1_expected_answers.json` from `ANNOTATIONS_PATH`.



In [ ]:
# ── Imports ────────────────────────────────────────────────────
import gc, time, json, os
import numpy as np
import pandas as pd
from PIL import Image
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor
from bert_score import score as bert_score_fn

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 200)

with open(f'{ANNOTATIONS_PATH}/questions.json') as f:
    questions_data = json.load(f)
with open(f'{ANNOTATIONS_PATH}/exp1_expected_answers.json') as f:
    expected_answers = json.load(f)

exp1_categories = questions_data['experiment1']['categories']
print('Imports OK')
print('Categories:', list(exp1_categories.keys()))

In [ ]:
# -- Metric functions -------------------------------------------

def normalize(text):
    return str(text).lower().strip()

def calc_exact_match(prediction, expected):
    """Robust exact match for closed-vocabulary and yes/no questions.

    Uses word-boundary matching to avoid substring leaks (so 'up' does not
    match 'upper'), and prioritises negation cues so responses like
    'there is no door' are not counted as positive.
    """
    import re
    pred = normalize(prediction)
    exp = normalize(expected)
    if not pred or not exp:
        return 0.0

    if exp in ('yes', 'no'):
        is_neg = bool(re.search(
            r"\bno\b|\bnot\b|\bnone\b|\bnever\b|\bcannot\b|\bcan't\b"
            r"|\bthere (?:is|are) no\b"
            r"|\bi (?:do not|don't|cannot|can't) see\b"
            r"|\bnot (?:visible|present)\b",
            pred,
        ))
        is_pos = bool(re.search(
            r"\byes\b|\byeah\b|\byep\b"
            r"|\bi (?:can see|see)\b"
            r"|\bthere (?:is|are) (?:a |an |some )?\w"
            r"|\b(?:visible|present)\b"
            r"|\bit is\b|\bthis is\b|\bthat is\b",
            pred,
        )) and not is_neg
        if exp == 'yes':
            return 1.0 if is_pos else 0.0
        return 1.0 if (is_neg and not is_pos) else 0.0

    tokens = pred.split()
    if tokens and tokens[0] == exp:
        return 1.0
    return 1.0 if re.search(r"\b" + re.escape(exp) + r"\b", pred) else 0.0


def calc_keyword_recall(prediction, keywords):
    pred = normalize(prediction)
    for kw in keywords:
        if normalize(kw) in pred:
            return 1.0
    return 0.0

def compute_confusion_matrix(results_df, category, question_id):
    df = results_df[
        (results_df['category'] == category) &
        (results_df['question_id'] == question_id) &
        (results_df['exact_match'].notna())
    ].copy()
    is_pos = df['expected'].str.lower() == 'yes'
    tp = int(len(df[is_pos  & (df['exact_match'] == 1.0)]))
    fn = int(len(df[is_pos  & (df['exact_match'] == 0.0)]))
    tn = int(len(df[~is_pos & (df['exact_match'] == 1.0)]))
    fp = int(len(df[~is_pos & (df['exact_match'] == 0.0)]))
    return tp, fp, tn, fn

def build_state_confusion(results_df, question_ids, labels):
    """Build a confusion matrix for state questions with a fixed label set.

    Looks for any label as a whole word inside the prediction, not just the
    first token, so responses like 'the doors are open' are classified
    correctly. Returns (matrix, correct, total, unknown_pred).
    """
    import re
    subset = results_df[results_df['question_id'].isin(question_ids)].copy()
    subset = subset[subset['prediction'].notna() & subset['expected'].notna()]

    label_to_idx = {l: i for i, l in enumerate(labels)}
    label_set = set(labels)
    n = len(labels)
    matrix = [[0] * n for _ in range(n)]
    correct, total, unknown_pred = 0, 0, 0

    for _, row in subset.iterrows():
        pred_text = normalize(str(row['prediction']))
        exp = normalize(str(row['expected']))

        ei = label_to_idx.get(exp)
        if ei is None:
            continue

        tokens = pred_text.split()
        first = tokens[0] if tokens else ''
        pred_label = first if first in label_set else None
        if pred_label is None:
            for label in labels:
                if re.search(r"\b" + re.escape(label) + r"\b", pred_text):
                    pred_label = label
                    break

        total += 1
        pi = label_to_idx.get(pred_label) if pred_label else None
        if pi is not None:
            matrix[ei][pi] += 1
            if pi == ei:
                correct += 1
        else:
            unknown_pred += 1

    accuracy = correct / total if total > 0 else 0.0
    return matrix, correct, total, unknown_pred, accuracy


# Canonical element vocabularies. ORDER IS LOAD-BEARING:
# sign is intentionally checked before door so compound phrases like
# 'bathroom door sign' match sign first rather than door via dict iteration.
ELEMENT_CANONICAL = {
    'staircase': ['stair', 'stairs', 'staircase', 'stairway', 'steps'],
    'elevator':  ['elevator', 'lift'],
    'sign':      ['sign', 'signal', 'notice', 'label', 'placard', 'plaque', 'board',
                  'bathroom', 'restroom', 'toilet', 'wc', 'exit', 'emergency'],
    'door':      ['door', 'doorway', 'entrance'],
    'obstacle':  ['obstacle', 'object', 'obstruction', 'barrier', 'block', 'box', 'cart',
                  'chair', 'table', 'rack', 'shelf', 'shelves', 'trolley', 'furniture',
                  'plant', 'bin', 'bucket', 'bag', 'luggage'],
    'corridor':  ['corridor', 'hallway', 'hall', 'passage', 'aisle'],
    'other':     ['other', 'none', 'unknown']
}
GT_ELEMENT_MAP = {
    'staircase': 'staircase', 'elevator': 'elevator', 'door': 'door',
    'door_sign': 'sign', 'obstacle': 'obstacle'
}
ELEMENT_QUESTION_IDS = {
    'staircase': 'staircase_element', 'elevator': 'elevator_element',
    'door': 'door_element', 'door_sign': 'doorsign_element', 'obstacle': 'obstacle_object'
}

def pred_to_canonical(prediction):
    """Map a free-text prediction to a canonical navigation element.

    Uses word-boundary regex so 'signal' does not leak into 'sign', and
    iterates ELEMENT_CANONICAL in declaration order (sign BEFORE door) so
    compound phrases like 'bathroom door sign' classify as sign.
    """
    import re
    pred = normalize(str(prediction))
    for canon, keywords in ELEMENT_CANONICAL.items():
        for kw in keywords:
            if re.search(r"\b" + re.escape(kw) + r"\b", pred):
                return canon
    return 'other'

print('Metric functions defined')

In [ ]:
# ── Plot helpers ───────────────────────────────────────────────

def save_heatmap(matrix, labels_row, labels_col, title, filepath, cmap='Blues', annot_fontsize=14):
    matrix_np = np.array(matrix)
    fig, ax = plt.subplots(figsize=(max(5, len(labels_col)*1.4), max(4, len(labels_row)*1.2)))
    im = ax.imshow(matrix_np, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(labels_col)))
    ax.set_yticks(range(len(labels_row)))
    ax.set_xticklabels(labels_col, fontsize=11)
    ax.set_yticklabels(labels_row, fontsize=11)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Ground Truth', fontsize=12)
    ax.set_title(title, fontsize=11, pad=10)
    thresh = matrix_np.max() / 2.0
    for i in range(len(labels_row)):
        for j in range(len(labels_col)):
            val = matrix_np[i, j]
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=annot_fontsize,
                    color='white' if val > thresh else 'black')
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()

def save_all_plots(df, agg_df, model_id, plots_path,
                   oc_matrix, oc_labels, ud_matrix, ud_labels,
                   el_matrix, el_labels):
    os.makedirs(plots_path, exist_ok=True)
    safe = model_id.replace('/', '_').replace('-', '_')

    # 1. open/closed
    save_heatmap(oc_matrix, oc_labels, oc_labels,
                 f'State: Open/Closed\n{model_id}',
                 f'{plots_path}/{safe}_state_open_closed.png')

    # 2. up/down
    save_heatmap(ud_matrix, ud_labels, ud_labels,
                 f'State: Up/Down\n{model_id}',
                 f'{plots_path}/{safe}_state_up_down.png')

    # 3. navigation elements
    save_heatmap(el_matrix, el_labels, el_labels,
                 f'Navigation Elements\n{model_id}',
                 f'{plots_path}/{safe}_nav_elements.png',
                 annot_fontsize=12)

    # 4. detection 2x2 per category
    det_cats = {
        'staircase': ('staircase_detection', 'Staircase Detection'),
        'elevator':  ('elevator_detection',  'Elevator Detection'),
        'door':      ('door_detection',      'Door Detection'),
        'obstacle':  ('obstacle_detection',  'Obstacle Detection'),
    }
    for cat, (qid, title) in det_cats.items():
        tp, fp, tn, fn = compute_confusion_matrix(df, cat, qid)
        save_heatmap([[tp, fn], [fp, tn]],
                     ['Actual Positive', 'Actual Negative'],
                     ['Pred Positive', 'Pred Negative'],
                     f'{title}\n{model_id}',
                     f'{plots_path}/{safe}_detection_{cat}.png', cmap='Greens')

    for det_id, title in [('doorsign_bathroom', 'Door Sign: Bathroom'),
                           ('doorsign_exit',     'Door Sign: Exit')]:
        tp, fp, tn, fn = compute_confusion_matrix(df, 'door_sign', det_id)
        save_heatmap([[tp, fn], [fp, tn]],
                     ['Actual Positive', 'Actual Negative'],
                     ['Pred Positive', 'Pred Negative'],
                     f'{title}\n{model_id}',
                     f'{plots_path}/{safe}_detection_{det_id}.png', cmap='Greens')

    # 5. obstacle safety 2x2
    tp_s, fp_s, tn_s, fn_s = compute_confusion_matrix(df, 'obstacle', 'obstacle_safety')
    save_heatmap([[tp_s, fn_s], [fp_s, tn_s]],
                 ['Actual Safe', 'Actual Unsafe'],
                 ['Pred Safe', 'Pred Unsafe'],
                 f'Obstacle Safety\n{model_id}',
                 f'{plots_path}/{safe}_safety_obstacle.png', cmap='Oranges')

    # 6. accuracy bar chart
    cats = agg_df['category'].tolist()
    accs = agg_df['accuracy'].tolist()
    colors = ['#4CAF50' if a >= 0.9 else '#FF9800' if a >= 0.7 else '#F44336' for a in accs]
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(cats, accs, color=colors)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Accuracy by Category\n{model_id}', fontsize=11)
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.3f}', ha='center', va='bottom', fontsize=10)
    patches = [mpatches.Patch(color='#4CAF50', label='>= 0.9'),
               mpatches.Patch(color='#FF9800', label='0.7-0.9'),
               mpatches.Patch(color='#F44336', label='< 0.7')]
    ax.legend(handles=patches, fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{plots_path}/{safe}_accuracy_by_category.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Plots saved to {plots_path}')

print('Plot helpers defined')

In [ ]:
# -- Single-model runner ----------------------------------------

def _make_synthetic_image(seed=SEED):
    """Return a 224x224 deterministic RGB-noise PIL image for warmup.

    Disjoint from the measured images so no real image benefits from a
    pre-warmed disk cache on its first timed run. Seeded for bit-reproducibility.
    """
    rng = np.random.default_rng(seed)
    arr = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)
    return Image.fromarray(arr, mode='RGB')


def run_experiment1_for_model(model_meta):
    """Runs full Exp1 for one model. Returns summary dict for comparison table.
    Saves all per-model files. Frees GPU memory at the end.
    """
    model_id   = model_meta['model_id']
    safe_name  = model_id.replace('/', '_').replace('-', '_')
    model_path = f'{RESULTS_PATH}/{safe_name}'
    plots_path = f'{model_path}/plots'
    os.makedirs(model_path, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'MODEL: {model_id}')
    print(f'{"="*60}')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    load_start = time.time()
    processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, token=HF_TOKEN,
        torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32
    ).to(device).eval()
    model_load_time_s = round(time.time() - load_start, 2)
    n_parameters = sum(p.numel() for p in model.parameters())
    print(f'  Loaded on {device} -- {n_parameters/1e9:.2f}B params in {model_load_time_s:.1f}s')

    def _sync():
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    def run_inference_timed(image, question):
        """Generate with the pinned protocol; CUDA-synchronized timing.

        Returns (prediction, latency_ms). Kwargs must stay identical to the
        DECODING_PROTOCOL dict in cell 5 for the repro card to remain truthful.
        """
        prompt = f'<image>{question}'
        inputs = processor(text=prompt, images=image, padding='longest',
                           do_convert_rgb=True, return_tensors='pt').to(device)
        inputs = inputs.to(dtype=model.dtype)
        _sync()
        t0 = time.time()
        with torch.no_grad():
            output = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, num_beams=1,
                pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
                eos_token_id=processor.tokenizer.eos_token_id
            )
        _sync()
        latency_ms = (time.time() - t0) * 1000
        input_len = inputs['input_ids'].shape[-1]
        pred = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()
        return pred, latency_ms

    def warmup_model():
        """Run N_WARMUP timed inferences and discard results. Pays cold-start
        cost once (kernel compilation, VRAM warmup, cache stabilization)."""
        warmup_img = _make_synthetic_image(SEED)
        for _ in range(N_WARMUP):
            run_inference_timed(warmup_img, 'What is in this image?')

    print(f'  Warming up ({N_WARMUP} iters on synthetic image, discarded)...')
    warmup_model()

    # -- Flatten the measured task list once -----------------------
    inference_tasks = []
    for category_name, category_data in exp1_categories.items():
        image_dir = os.path.join(IMAGES_PATH, category_name)
        if not os.path.exists(image_dir):
            print(f'  WARNING: folder not found: {image_dir}')
            continue
        images = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
        cat_answers = expected_answers.get(category_name, {})

        for img_name in images:
            is_negative = '_neg_' in img_name
            img_answers = cat_answers.get(img_name, {})
            if not img_answers:
                continue
            for q_data in category_data['questions']:
                q_id = q_data['id']
                metrics = q_data['metrics']
                q_type  = q_data['type']
                if is_negative and 'tp_fp_tn_fn' not in metrics and q_type != 'element':
                    continue
                expected = img_answers.get(q_data['expected_answer_key'])
                if expected is None:
                    continue
                inference_tasks.append({
                    'category': category_name, 'img_name': img_name,
                    'is_negative': is_negative, 'q_data': q_data,
                    'img_answers': img_answers, 'expected': expected,
                })

    n_tasks = len(inference_tasks)
    print(f'  Tasks to run: {n_tasks} per latency run x {N_LATENCY_RUNS} runs')

    # -- Multi-run inference. Predictions only taken from run 0.
    # -- Per-run shuffle averages out any thermal drift across positions.
    predictions = [None] * n_tasks
    latencies = [[] for _ in range(n_tasks)]
    total_start = time.time()

    for run_idx in range(N_LATENCY_RUNS):
        order = np.arange(n_tasks)
        np.random.default_rng(SEED * 1000 + run_idx).shuffle(order)
        for pos, task_idx in enumerate(order):
            task = inference_tasks[task_idx]
            image = Image.open(os.path.join(IMAGES_PATH, task['category'], task['img_name'])).convert('RGB')
            pred, lat_ms = run_inference_timed(image, task['q_data']['question'])
            latencies[task_idx].append(lat_ms)
            if run_idx == 0:
                predictions[task_idx] = pred

    total_time_s = time.time() - total_start

    # -- Build stable-order result rows with per-run latency columns --
    results = []
    for task_idx, task in enumerate(inference_tasks):
        q_data = task['q_data']
        metrics = q_data['metrics']
        img_answers = task['img_answers']
        expected = task['expected']
        prediction = predictions[task_idx] if predictions[task_idx] is not None else ''
        lats = latencies[task_idx]
        lat_mean = float(np.mean(lats)) if lats else 0.0
        lat_std = float(np.std(lats, ddof=1)) if len(lats) >= 2 else 0.0
        row = {
            'model_id': model_meta['model_id'],
            'category': task['category'], 'image': task['img_name'],
            'is_negative': task['is_negative'], 'question_id': q_data['id'],
            'question_type': q_data['type'], 'question': q_data['question'],
            'prediction': prediction, 'expected': str(expected),
            'inference_time_ms': round(lat_mean, 2),
            'latency_ms_mean': round(lat_mean, 2),
            'latency_ms_std': round(lat_std, 2),
            'output_length_words': len(prediction.split()),
            'exact_match': None, 'keyword_recall': None,
            'bertscore_f1': None, 'bertscore_p': None, 'bertscore_r': None,
            'bertscore_reference': None,
        }
        for i, lat in enumerate(lats, start=1):
            row[f'latency_ms_run{i}'] = round(lat, 2)
        if 'exact_match' in metrics:
            row['exact_match'] = calc_exact_match(prediction, str(expected))
        if 'keyword_recall' in metrics:
            kws = img_answers.get('element_keywords', q_data.get('expected_keywords', []))
            row['keyword_recall'] = calc_keyword_recall(prediction, kws)
        if 'bertscore' in metrics:
            ref = img_answers.get('bertscore_reference', q_data.get('bertscore_reference', ''))
            if ref:
                row['bertscore_reference'] = ref
        results.append(row)

    df = pd.DataFrame(results)
    print(f'  Inferences: {len(df)} unique tasks, {len(df)*N_LATENCY_RUNS} timed calls in {total_time_s:.1f}s')

    # -- BERTScore ---------------------------------------------
    bs_mask = df['bertscore_reference'].notna() & (df['bertscore_reference'] != '')
    if bs_mask.sum() > 0:
        P, R, F1 = bert_score_fn(df[bs_mask]['prediction'].tolist(),
                                  df[bs_mask]['bertscore_reference'].tolist(),
                                  lang='en', verbose=False)
        df.loc[bs_mask, 'bertscore_f1'] = F1.numpy()
        df.loc[bs_mask, 'bertscore_p']  = P.numpy()
        df.loc[bs_mask, 'bertscore_r']  = R.numpy()

    # -- Aggregate metrics -------------------------------------
    agg_rows = []
    for category_name in exp1_categories.keys():
        cat_df = df[df['category'] == category_name]
        agg = {'category': category_name}
        em_df = cat_df[cat_df['exact_match'].notna()]
        if len(em_df):
            agg['accuracy'] = round(em_df['exact_match'].mean(), 4)

        if category_name == 'door_sign':
            for det_id in ['doorsign_bathroom', 'doorsign_exit']:
                if det_id in cat_df['question_id'].values:
                    tp, fp, tn, fn = compute_confusion_matrix(df, category_name, det_id)
                    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
                    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
                    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
                    k = det_id.replace('doorsign_', '')
                    agg.update({f'{k}_tp':tp, f'{k}_fp':fp, f'{k}_tn':tn, f'{k}_fn':fn,
                                f'{k}_precision':round(prec,4), f'{k}_recall':round(rec,4),
                                f'{k}_f1':round(f1,4)})
        else:
            det_id = f'{category_name}_detection'
            if det_id in cat_df['question_id'].values:
                tp, fp, tn, fn = compute_confusion_matrix(df, category_name, det_id)
                prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
                rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
                f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
                agg.update({'detection_tp':tp,'detection_fp':fp,'detection_tn':tn,'detection_fn':fn,
                            'detection_precision':round(prec,4),'detection_recall':round(rec,4),
                            'detection_f1':round(f1,4)})
                if category_name == 'obstacle':
                    agg['hazard_miss_rate'] = round(1-rec, 4)
                    if 'obstacle_safety' in cat_df['question_id'].values:
                        tp_s,fp_s,tn_s,fn_s = compute_confusion_matrix(df,'obstacle','obstacle_safety')
                        prec_s = tp_s/(tp_s+fp_s) if (tp_s+fp_s)>0 else 0.0
                        rec_s  = tp_s/(tp_s+fn_s) if (tp_s+fn_s)>0 else 0.0
                        f1_s   = 2*prec_s*rec_s/(prec_s+rec_s) if (prec_s+rec_s)>0 else 0.0
                        safety_hmr = round(fp_s/(fp_s+tn_s), 4) if (fp_s+tn_s)>0 else 0.0
                        agg.update({'safety_tp':tp_s,'safety_fp':fp_s,'safety_tn':tn_s,'safety_fn':fn_s,
                                    'safety_precision':round(prec_s,4),'safety_recall':round(rec_s,4),
                                    'safety_f1':round(f1_s,4),
                                    'safety_hazard_miss_rate':safety_hmr})

        kr_df = cat_df[cat_df['keyword_recall'].notna()]
        if len(kr_df):
            agg['keyword_recall'] = round(kr_df['keyword_recall'].mean(), 4)
        bs_df = cat_df[cat_df['bertscore_f1'].notna()]
        if len(bs_df):
            agg['bertscore_f1'] = round(bs_df['bertscore_f1'].mean(), 4)
            agg['bertscore_p']  = round(bs_df['bertscore_p'].mean(), 4)
            agg['bertscore_r']  = round(bs_df['bertscore_r'].mean(), 4)
        agg['avg_output_length'] = round(cat_df['output_length_words'].mean(), 2)
        agg_rows.append(agg)

    agg_df = pd.DataFrame(agg_rows)

    # -- Efficiency --------------------------------------------
    vram_peak = torch.cuda.max_memory_allocated()/(1024**3) if device=='cuda' else 0
    n_images  = df['image'].nunique()
    total_inf_ms = df['inference_time_ms'].sum()
    efficiency = {
        'model_id': model_id, 'experiment': 'exp1',
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'n_images': n_images, 'n_inferences': len(df),
        'vram_peak_gb': round(vram_peak, 3),
        'model_load_time_s': model_load_time_s,
        'total_wall_time_s': round(total_time_s, 2),
        'total_inference_time_s': round(total_inf_ms / 1000, 2),
        'avg_inference_time_ms': round(df['inference_time_ms'].mean(), 2),
        'avg_inference_time_ms_std': round(df['latency_ms_std'].mean(), 2),
        'throughput_img_per_s': round(n_images/(total_inf_ms/1000), 3) if total_inf_ms>0 else 0,
        'throughput_inf_per_s': round(len(df)/(total_inf_ms/1000), 3) if total_inf_ms>0 else 0,
        'avg_output_length_words': round(df['output_length_words'].mean(), 2),
        'max_output_length_words': int(df['output_length_words'].max())
    }

    # -- State matrices ----------------------------------------
    oc_matrix, oc_correct, oc_total, oc_unk, oc_acc = build_state_confusion(
        df, ['elevator_doors','door_state','doorsign_state'], ['open','closed'])
    ud_matrix, ud_correct, ud_total, ud_unk, ud_acc = build_state_confusion(
        df, ['staircase_direction'], ['up','down'])
    oc_labels = ['open','closed']
    ud_labels = ['up','down']

    # -- Navigation elements matrix ----------------------------
    el_labels = ['staircase','elevator','door','sign','obstacle','corridor','other']
    el_idx    = {l:i for i,l in enumerate(el_labels)}
    n = len(el_labels)
    el_matrix = [[0]*n for _ in range(n)]
    el_correct, el_total = 0, 0
    valid_qids = set(ELEMENT_QUESTION_IDS.values())
    el_df = df[(df['question_id'].isin(valid_qids)) & (~df['is_negative'])].copy()
    for _, row in el_df.iterrows():
        gt = GT_ELEMENT_MAP.get(row['category'])
        if gt is None: continue
        pr = pred_to_canonical(row['prediction'])
        ei, pi = el_idx.get(gt), el_idx.get(pr)
        if ei is None or pi is None: continue
        el_matrix[ei][pi] += 1
        el_total += 1
        if ei == pi: el_correct += 1
    el_acc = el_correct/el_total if el_total>0 else 0.0

    matrices = {
        'model_id': model_id, 'experiment': 'exp1',
        'state_open_closed': {
            'labels': oc_labels, 'matrix': oc_matrix,
            'accuracy': round(oc_acc,4), 'correct': oc_correct,
            'total': oc_total, 'unknown_predictions': oc_unk,
            'question_ids': ['elevator_doors','door_state','doorsign_state'],
            'note': 'Rows = ground truth, columns = predicted'
        },
        'state_up_down': {
            'labels': ud_labels, 'matrix': ud_matrix,
            'accuracy': round(ud_acc,4), 'correct': ud_correct,
            'total': ud_total, 'unknown_predictions': ud_unk,
            'question_ids': ['staircase_direction'],
            'note': 'Rows = ground truth, columns = predicted'
        },
        'navigation_elements': {
            'labels': el_labels, 'matrix': el_matrix,
            'accuracy': round(el_acc,4), 'correct': el_correct,
            'total': el_total, 'gt_element_map': GT_ELEMENT_MAP,
            'note': 'Rows = ground truth category, columns = predicted canonical element'
        }
    }

    # -- Save files --------------------------------------------
    df.to_csv(f'{model_path}/results.csv', index=False)
    agg_df.to_csv(f'{model_path}/aggregate.csv', index=False)
    with open(f'{model_path}/efficiency.json','w') as f_out:
        json.dump(efficiency, f_out, indent=2)
    with open(f'{model_path}/matrices.json','w') as f_out:
        json.dump(matrices, f_out, indent=2)

    save_all_plots(df, agg_df, model_id, plots_path,
                   oc_matrix, oc_labels, ud_matrix, ud_labels,
                   el_matrix, el_labels)

    print(f'  All files saved to {model_path}')

    # -- Build summary dict for comparison table ---------------
    def agg_val(cat, col):
        row = agg_df[agg_df['category']==cat]
        return float(row[col].values[0]) if len(row) and col in row.columns and pd.notna(row[col].values[0]) else None

    summary = {
        'model_id':   model_id,
        'version':    model_meta.get('version'),
        'size':       model_meta.get('size'),
        'type':       model_meta.get('type'),
        'resolution': model_meta.get('resolution'),
        'staircase_accuracy':   agg_val('staircase','accuracy'),
        'elevator_accuracy':    agg_val('elevator','accuracy'),
        'door_accuracy':        agg_val('door','accuracy'),
        'door_sign_accuracy':   agg_val('door_sign','accuracy'),
        'obstacle_accuracy':    agg_val('obstacle','accuracy'),
        'staircase_detection_f1': agg_val('staircase','detection_f1'),
        'elevator_detection_f1':  agg_val('elevator','detection_f1'),
        'door_detection_f1':      agg_val('door','detection_f1'),
        'door_detection_precision': agg_val('door','detection_precision'),
        'door_detection_recall':    agg_val('door','detection_recall'),
        'obstacle_detection_f1':  agg_val('obstacle','detection_f1'),
        'doorsign_bathroom_f1': agg_val('door_sign','bathroom_f1'),
        'doorsign_exit_f1':     agg_val('door_sign','exit_f1'),
        'hazard_miss_rate':        agg_val('obstacle','hazard_miss_rate'),
        'safety_hazard_miss_rate': agg_val('obstacle','safety_hazard_miss_rate'),
        'obstacle_safety_f1':      agg_val('obstacle','safety_f1'),
        'obstacle_safety_precision': agg_val('obstacle','safety_precision'),
        'obstacle_safety_recall':    agg_val('obstacle','safety_recall'),
        'staircase_keyword_recall':  agg_val('staircase','keyword_recall'),
        'elevator_keyword_recall':   agg_val('elevator','keyword_recall'),
        'door_keyword_recall':       agg_val('door','keyword_recall'),
        'door_sign_keyword_recall':  agg_val('door_sign','keyword_recall'),
        'obstacle_keyword_recall':   agg_val('obstacle','keyword_recall'),
        'staircase_bertscore_f1':  agg_val('staircase','bertscore_f1'),
        'elevator_bertscore_f1':   agg_val('elevator','bertscore_f1'),
        'door_bertscore_f1':       agg_val('door','bertscore_f1'),
        'door_sign_bertscore_f1':  agg_val('door_sign','bertscore_f1'),
        'obstacle_bertscore_f1':   agg_val('obstacle','bertscore_f1'),
        'state_open_closed_acc': round(oc_acc,4),
        'state_up_down_acc':     round(ud_acc,4),
        'nav_elements_acc':      round(el_acc,4),
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'vram_peak_gb':          efficiency['vram_peak_gb'],
        'model_load_time_s':     model_load_time_s,
        'avg_inference_time_ms': efficiency['avg_inference_time_ms'],
        'avg_inference_time_ms_std': efficiency['avg_inference_time_ms_std'],
        'throughput_img_per_s':  efficiency['throughput_img_per_s'],
        'throughput_inf_per_s':  efficiency['throughput_inf_per_s'],
        'avg_output_length_words': efficiency['avg_output_length_words'],
    }

    acc_vals = [summary[f'{c}_accuracy'] for c in
                ['staircase','elevator','door','door_sign','obstacle']
                if summary.get(f'{c}_accuracy') is not None]
    summary['mean_accuracy'] = round(sum(acc_vals)/len(acc_vals), 4) if acc_vals else None

    bs_vals = [summary[f'{c}_bertscore_f1'] for c in
               ['staircase','elevator','door','door_sign','obstacle']
               if summary.get(f'{c}_bertscore_f1') is not None]
    summary['mean_bertscore_f1'] = round(sum(bs_vals)/len(bs_vals), 4) if bs_vals else None

    kr_vals = [summary[f'{c}_keyword_recall'] for c in
               ['staircase','elevator','door','door_sign','obstacle']
               if summary.get(f'{c}_keyword_recall') is not None]
    summary['mean_keyword_recall'] = round(sum(kr_vals)/len(kr_vals), 4) if kr_vals else None

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'  GPU memory freed')

    return summary

print('Single-model runner defined')

In [ ]:
# ── Main loop — run all models ─────────────────────────────────
all_summaries = []
failed_models = []

for model_meta in run_models:
    model_id = model_meta['model_id']
    try:
        summary = run_experiment1_for_model(model_meta)
        all_summaries.append(summary)
        print(f'  OK: {model_id}  mean_accuracy={summary["mean_accuracy"]}')
    except Exception as e:
        print(f'  FAILED: {model_id} — {e}')
        failed_models.append({'model_id': model_id, 'error': str(e)})
        # Free memory even on failure
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

print(f'\nCompleted: {len(all_summaries)}/{len(run_models)} models')
if failed_models:
    print('Failed:', [m["model_id"] for m in failed_models])

In [ ]:
# ── Cross-model comparison table ───────────────────────────────
comp_df = pd.DataFrame(all_summaries)

# ── Summary view (key metrics only) ───────────────────────────
summary_cols = [
    'model_id', 'version', 'size', 'type', 'resolution',
    'n_parameters_billions',
    'mean_accuracy', 'mean_bertscore_f1', 'mean_keyword_recall',
    'staircase_accuracy', 'elevator_accuracy', 'door_accuracy',
    'door_sign_accuracy', 'obstacle_accuracy',
    'hazard_miss_rate', 'safety_hazard_miss_rate',
    'state_open_closed_acc', 'state_up_down_acc', 'nav_elements_acc',
    'vram_peak_gb', 'model_load_time_s', 'avg_inference_time_ms',
    'throughput_img_per_s', 'throughput_inf_per_s'
]
summary_cols = [c for c in summary_cols if c in comp_df.columns]
summary_df = comp_df[summary_cols].copy()

print('ALL MODELS — SUMMARY')
print('=' * 80)
display(summary_df.round(4))

### 6.5 Statistical significance -- bootstrap CIs and pairwise McNemar tests

In [ ]:
# -- Bootstrap 95% CIs and pairwise McNemar tests ---------------
from scipy.stats import binomtest

N_BOOTSTRAP = 1000
rng = np.random.default_rng(SEED)


def bootstrap_ci(values, agg_fn=np.mean, n_boot=N_BOOTSTRAP, ci=95):
    """Return (mean, lo, hi) of agg_fn(values) with a percentile bootstrap CI.
    Drops NaN before resampling. Returns (nan, nan, nan) if no finite values remain.
    """
    arr = np.asarray([v for v in values if v is not None and np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float('nan'), float('nan'), float('nan')
    idx = rng.integers(0, arr.size, size=(n_boot, arr.size))
    samples = agg_fn(arr[idx], axis=1)
    lo = float(np.percentile(samples, (100 - ci) / 2))
    hi = float(np.percentile(samples, 100 - (100 - ci) / 2))
    return float(agg_fn(arr)), lo, hi


def holm_bonferroni(pvals):
    """Holm step-down adjusted p-values. Controls family-wise error at alpha
    without assuming independence across tests.
    """
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    adj = np.empty(n)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (n - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj


# Rebuild per-model per-item exact_match vectors from the per-model CSVs so
# this cell is self-contained even if the main loop's all_summaries list is stale.
per_model_em = {}
for safe_dir in sorted(os.listdir(RESULTS_PATH)):
    csv_path = os.path.join(RESULTS_PATH, safe_dir, 'results.csv')
    if not os.path.isfile(csv_path):
        continue
    mdf = pd.read_csv(csv_path)
    mdf = mdf[mdf['exact_match'].notna()].copy()
    if mdf.empty:
        continue
    mdf['key'] = mdf['image'].astype(str) + '::' + mdf['question_id'].astype(str)
    per_model_em[safe_dir] = dict(zip(mdf['key'], mdf['exact_match'].astype(float)))

model_safe_names = sorted(per_model_em.keys())
if len(model_safe_names) < 2:
    print('Need at least 2 models with results.csv to run significance tests.')
    ci_df = pd.DataFrame()
    pairs_df = pd.DataFrame()
else:
    # Intersect keys so every model is evaluated on the same items.
    common_keys = set.intersection(*[set(d.keys()) for d in per_model_em.values()])
    common_keys = sorted(common_keys)
    print(f'Common items across {len(model_safe_names)} models: {len(common_keys)}')

    em_matrix = np.array([
        [per_model_em[m][k] for k in common_keys] for m in model_safe_names
    ], dtype=float)

    # -- 1. Bootstrap 95% CIs on per-model exact_match mean ------
    ci_rows = []
    for i, m in enumerate(model_safe_names):
        mean, lo, hi = bootstrap_ci(em_matrix[i] * 100)
        ci_rows.append({
            'model_safe': m,
            'n_items': int(em_matrix.shape[1]),
            'exact_match_mean_pct': round(mean, 2) if np.isfinite(mean) else None,
            'exact_match_ci_low_pct': round(lo, 2) if np.isfinite(lo) else None,
            'exact_match_ci_high_pct': round(hi, 2) if np.isfinite(hi) else None,
        })
    ci_df = pd.DataFrame(ci_rows).sort_values('exact_match_mean_pct', ascending=False).reset_index(drop=True)
    ci_df.to_csv(f'{RESULTS_PATH}/bootstrap_cis.csv', index=False)
    print(f'Saved: {RESULTS_PATH}/bootstrap_cis.csv ({N_BOOTSTRAP} resamples per row)')

    # -- 2. Pairwise McNemar (exact binomial on discordant pairs) --
    pairs = []
    raw_p = []
    for i, a in enumerate(model_safe_names):
        for j in range(i + 1, len(model_safe_names)):
            b = model_safe_names[j]
            va = em_matrix[i]
            vb = em_matrix[j]
            b_only = int(((va == 0) & (vb == 1)).sum())
            a_only = int(((va == 1) & (vb == 0)).sum())
            n_disc = b_only + a_only
            if n_disc == 0:
                pvalue = 1.0
            else:
                res = binomtest(min(b_only, a_only), n_disc, 0.5, alternative='two-sided')
                pvalue = float(res.pvalue)
            mean_a, mean_b = float(va.mean()), float(vb.mean())
            better = a if mean_a > mean_b else b
            pairs.append({
                'model_a': a, 'model_b': b,
                'mean_em_a_pct': round(mean_a * 100, 2),
                'mean_em_b_pct': round(mean_b * 100, 2),
                'better_model': better,
                'p_value_raw': round(pvalue, 5),
            })
            raw_p.append(pvalue)

    for row, adj in zip(pairs, holm_bonferroni(raw_p)):
        row['p_value_holm'] = round(float(adj), 5)
        row['significant_0_05'] = bool(adj < 0.05)

    pairs_df = pd.DataFrame(pairs).sort_values('p_value_holm').reset_index(drop=True)
    pairs_df.to_csv(f'{RESULTS_PATH}/statistical_tests.csv', index=False)
    n_sig = int(pairs_df['significant_0_05'].sum())
    print(f'Saved: {RESULTS_PATH}/statistical_tests.csv ({len(pairs_df)} pairs, Holm-corrected)')
    print(f'Significant pairs (Holm p<0.05): {n_sig}/{len(pairs_df)}')

In [ ]:
# -- Horizontal bar plot: exact_match ranking with bootstrap CIs
comp_plots_path = f'{RESULTS_PATH}/comparison_plots'
os.makedirs(comp_plots_path, exist_ok=True)

if not ci_df.empty:
    sorted_ci = ci_df.sort_values('exact_match_mean_pct', ascending=False).reset_index(drop=True)
    means = sorted_ci['exact_match_mean_pct'].values.astype(float)
    lows = means - sorted_ci['exact_match_ci_low_pct'].values.astype(float)
    highs = sorted_ci['exact_match_ci_high_pct'].values.astype(float) - means

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(sorted_ci))))
    y = np.arange(len(sorted_ci))
    ax.barh(
        y, means, xerr=[lows, highs],
        color='#4CAF50', ecolor='black',
        capsize=3, error_kw={'linewidth': 0.8}, alpha=0.85,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(sorted_ci['model_safe'].values, fontsize=9)
    ax.invert_yaxis()  # best model on top
    ax.set_xlabel(f'Exact match (%)  --  error bars = 95% bootstrap CI over {N_BOOTSTRAP} resamples')
    ax.set_title('Exact-match ranking with confidence intervals')
    ax.grid(axis='x', alpha=0.3)
    pad = (max(highs) if len(highs) else 1.0) * 0.05
    for i, m in enumerate(means):
        if np.isfinite(m):
            ax.text(m + pad, i, f'{m:.2f}', va='center', fontsize=8)
    plt.tight_layout()
    out = f'{comp_plots_path}/cmp_exact_match_with_cis.png'
    plt.savefig(out, dpi=120)
    plt.close()
    print(f'Saved: {out}')
else:
    print('Skipping CI plot: ci_df is empty.')

In [ ]:
# ── Cross-model comparison plots ───────────────────────────────
comp_plots_path = f'{RESULTS_PATH}/comparison_plots'
os.makedirs(comp_plots_path, exist_ok=True)

model_labels = [m.replace('google/', '').replace('paligemma', 'pg') for m in comp_df['model_id']]

def comparison_bar(col, title, ylabel, filename, higher_is_better=True):
    if col not in comp_df.columns:
        return
    vals = comp_df[col].tolist()
    if all(v is None for v in vals):
        return
    fig, ax = plt.subplots(figsize=(max(8, len(vals)*1.2), 4))
    colors = []
    for v in vals:
        if v is None:
            colors.append('#CCCCCC')
        elif higher_is_better:
            colors.append('#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.7 else '#F44336')
        else:  # lower is better (HMR)
            colors.append('#4CAF50' if v <= 0.1 else '#FF9800' if v <= 0.3 else '#F44336')
    bars = ax.bar(model_labels, [v if v is not None else 0 for v in vals], color=colors)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xticklabels(model_labels, rotation=30, ha='right', fontsize=8)
    for bar, val in zip(bars, vals):
        if val is not None:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.savefig(f'{comp_plots_path}/{filename}', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {filename}')

# Accuracy plots
comparison_bar('mean_accuracy',    'Mean Accuracy — All Models',        'Accuracy', 'cmp_mean_accuracy.png')
comparison_bar('staircase_accuracy','Staircase Accuracy — All Models',  'Accuracy', 'cmp_staircase_accuracy.png')
comparison_bar('elevator_accuracy', 'Elevator Accuracy — All Models',   'Accuracy', 'cmp_elevator_accuracy.png')
comparison_bar('door_accuracy',     'Door Accuracy — All Models',       'Accuracy', 'cmp_door_accuracy.png')
comparison_bar('obstacle_accuracy', 'Obstacle Accuracy — All Models',   'Accuracy', 'cmp_obstacle_accuracy.png')

# Safety
comparison_bar('hazard_miss_rate',        'Hazard Miss Rate (Detection) — All Models',
               'HMR (lower=safer)', 'cmp_hazard_miss_rate.png', higher_is_better=False)
comparison_bar('safety_hazard_miss_rate', 'Safety: false clear-path rate (obstacle_safety) — All Models',
               'FP/(FP+TN), lower=safer', 'cmp_safety_hmr.png', higher_is_better=False)

# BERTScore & keyword
comparison_bar('mean_bertscore_f1',   'Mean BERTScore F1 — All Models',     'BERTScore F1', 'cmp_mean_bertscore.png')
comparison_bar('mean_keyword_recall', 'Mean Keyword Recall — All Models',   'Keyword Recall','cmp_mean_keyword.png')

# State matrices
comparison_bar('state_open_closed_acc', 'State Open/Closed Accuracy — All Models', 'Accuracy', 'cmp_state_oc.png')
comparison_bar('state_up_down_acc',     'State Up/Down Accuracy — All Models',     'Accuracy', 'cmp_state_ud.png')
comparison_bar('nav_elements_acc',      'Navigation Elements Accuracy — All Models','Accuracy','cmp_nav_elements.png')

# Efficiency (VRAM)
if 'vram_peak_gb' in comp_df.columns:
    vals = comp_df['vram_peak_gb'].tolist()
    fig, ax = plt.subplots(figsize=(max(8, len(vals)*1.2), 4))
    ax.bar(model_labels, vals, color='#5C6BC0')
    ax.set_ylabel('VRAM Peak (GB)', fontsize=11)
    ax.set_title('VRAM Peak — All Models', fontsize=12)
    ax.set_xticklabels(model_labels, rotation=30, ha='right', fontsize=8)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{comp_plots_path}/cmp_vram.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved: cmp_vram.png')

print(f'\nAll comparison plots saved to {comp_plots_path}')

In [ ]:
# -- Save comparison CSVs and final report ----------------------
summary_df.to_csv(f'{RESULTS_PATH}/all_models_summary.csv', index=False)
comp_df.to_csv(f'{RESULTS_PATH}/all_models_expanded.csv', index=False)

if failed_models:
    with open(f'{RESULTS_PATH}/failed_models.json','w') as f:
        json.dump(failed_models, f, indent=2)

print('FINAL OUTPUT')
print('=' * 60)
print(f'  all_models_summary.csv  -- key metrics, {len(summary_df)} models')
print(f'  all_models_expanded.csv -- all {len(comp_df.columns)} columns, {len(comp_df)} models')
print(f'  exp1_reproducibility.json')
print(f'  {RESULTS_PATH}/bootstrap_cis.csv')
print(f'  {RESULTS_PATH}/statistical_tests.csv')
print(f'  {RESULTS_PATH}/comparison_plots/cmp_exact_match_with_cis.png')
if failed_models:
    print(f'  failed_models.json -- {len(failed_models)} failed')
print(f'\nDownload /content/results/exp1/ as ZIP when done.')
print(f'\nBEST MODEL by mean_accuracy:')
if len(summary_df) > 0 and 'mean_accuracy' in summary_df.columns:
    best = summary_df.sort_values('mean_accuracy', ascending=False).iloc[0]
    print(f'  {best["model_id"]}  mean_accuracy={best["mean_accuracy"]:.4f}')
print(f'\nSAFEST MODEL by hazard_miss_rate:')
if 'hazard_miss_rate' in summary_df.columns:
    safest = summary_df.dropna(subset=['hazard_miss_rate']).sort_values('hazard_miss_rate').iloc[0]
    print(f'  {safest["model_id"]}  hazard_miss_rate={safest["hazard_miss_rate"]:.4f}')